**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).

---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# CS231n local environment setup (exercise code unchanged)
# [中文] CS231n 本地环境设置 (exercise code unchanged)
from pathlib import Path
import os
import sys

_candidates = [Path.cwd(), *Path.cwd().parents]
_workspace = next((p for p in _candidates if (p / "requirements-local.txt").is_file()
                   and (p / "assignment1" / "cs231n").is_dir()), None)
if _workspace is None:
    raise RuntimeError("Launch Jupyter from the CS231n workspace root.")
ASSIGNMENT_DIR = _workspace / "assignment1"
os.chdir(ASSIGNMENT_DIR)
if str(ASSIGNMENT_DIR) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_DIR))
print("Python:", sys.executable)
print("Working directory:", Path.cwd())


# Image features exercise
*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission. For more details see the [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html) on the course website.*

We have seen that we can achieve reasonable performance on an image classification task by training a linear classifier on the pixels of the input image. In this exercise we will show that we can improve our classification performance by training linear classifiers not on raw pixels but on features that are computed from the raw pixels.

All of your work for this exercise will be done in this notebook.

---

**中文对照（参考译文，不含解答）**

# 图像特性练习
* 完成并提交已完成的工作表(including its outputs and any supporting code outside of the worksheet)。 详情见课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。 *

我们已经看到,通过对输入图像像素的线性分类师进行训练,我们就可以在图像分类任务上取得合理的业绩。 在此练习中,我们将显示,我们可以通过对线性分类员进行训练,而不是对原始像素,而是对从原始像素中计算出来的特征进行训练,从而改进我们的分类性能。

你为这次演习所做的一切工作 都会在这个笔记本里完成

In [ ]:
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt


%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
# [中文] 设置默认的绘图大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# [中文] 自动再装入前端模块时,请查看http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## Load data
Similar to previous exercises, we will load CIFAR-10 data from disk.

---

**中文对照（参考译文，不含解答）**

## 装入数据
与以往的练习类似,我们将从磁盘上装载CIFAR-10数据。

In [ ]:
from cs231n.features import color_histogram_hsv, hog_feature

def get_CIFAR10_data(num_training=49000, num_validation=1000, num_test=1000):
    # Load the raw CIFAR-10 data
    # [中文] 装入原始的CIFAR - 10 数据
    cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

    # Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
    # [中文] 清理变量以防止多次装入数据 (which may cause memory issue)
    try:
       del X_train, y_train
       del X_test, y_test
       print('Clear previously loaded data.')
    except:
       pass

    X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

    # Subsample the data
    # [中文] 分样数据
    mask = list(range(num_training, num_training + num_validation))
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = list(range(num_training))
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = list(range(num_test))
    X_test = X_test[mask]
    y_test = y_test[mask]

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = get_CIFAR10_data()

## Extract Features
For each image we will compute a Histogram of Oriented
Gradients (HOG) as well as a color histogram using the hue channel in HSV
color space. We form our final feature vector for each image by concatenating
the HOG and color histogram feature vectors.

Roughly speaking, HOG should capture the texture of the image while ignoring
color information, and the color histogram represents the color of the input
image while ignoring texture. As a result, we expect that using both together
ought to work better than using either alone. Verifying this assumption would
be a good thing to try for your own interest.

The `hog_feature` and `color_histogram_hsv` functions both operate on a single
image and return a feature vector for that image. The extract_features
function takes a set of images and a list of feature functions and evaluates
each feature function on each image, storing the results in a matrix where
each column is the concatenation of all feature vectors for a single image.

---

**中文对照（参考译文，不含解答）**

## 提取特性
对于每张图像,我们将计算一个方向直方图
梯度 (HOG) 以及使用 HSV 色谱频道的彩色直方图
颜色空格。 我们通过连接形成每种图像的最终特征向量
HOG 和 彩色直方图 特征向量。

大致上说,HOG应该捕捉图像的纹理,而忽略
颜色信息, 颜色直方图代表输入的颜色
图像却忽略了纹理。 因此,我们期望两者同时使用
应该比单独使用两者都有效。
为了你自己的利益 做个好尝试吧 \fn微软雅黑I be a good thing to try for yourself interest.

`hog_feature` 和 `color_histogram_hsv` 函数同时在单线操作
图像并返回该图像的特征向量。 extract_features
函数需要一组图像和特征函数列表并进行评估
每个图像的每个特性特征,将结果存储在矩阵中
每个列是单个图像中所有特征向量的集合。

In [ ]:
from cs231n.features import *

# num_color_bins = 10 # Number of bins in the color histogram
num_color_bins = 25 # Number of bins in the color histogram
# [中文] 颜色直方图中的文件夹数
feature_fns = [hog_feature, lambda img: color_histogram_hsv(img, nbin=num_color_bins)]
X_train_feats = extract_features(X_train, feature_fns, verbose=True)
X_val_feats = extract_features(X_val, feature_fns)
X_test_feats = extract_features(X_test, feature_fns)

# Preprocessing: Subtract the mean feature
# [中文] 预处理: 减去平均值
mean_feat = np.mean(X_train_feats, axis=0, keepdims=True)
X_train_feats -= mean_feat
X_val_feats -= mean_feat
X_test_feats -= mean_feat

# Preprocessing: Divide by standard deviation. This ensures that each feature
# has roughly the same scale.
# [中文] 预处理:除以标准差。 这将确保每个特性的大小大致相同。
std_feat = np.std(X_train_feats, axis=0, keepdims=True)
X_train_feats /= std_feat
X_val_feats /= std_feat
X_test_feats /= std_feat

# Preprocessing: Add a bias dimension
# [中文] 预处理: 添加偏差维度
X_train_feats = np.hstack([X_train_feats, np.ones((X_train_feats.shape[0], 1))])
X_val_feats = np.hstack([X_val_feats, np.ones((X_val_feats.shape[0], 1))])
X_test_feats = np.hstack([X_test_feats, np.ones((X_test_feats.shape[0], 1))])

## Train Softmax classifier on features
Using the Softmax code developed earlier in the assignment, train Softmax classifiers on top of the features extracted above; this should achieve better results than training them directly on top of raw pixels.

---

**中文对照（参考译文，不含解答）**

## 有关特性的软体分类器
利用任务中早些时候开发的软体成像码,在以上提取的特征之上训练软体分类人员;这比在生像素之上直接训练他们的结果要好。

In [ ]:
# Use the validation set to tune the learning rate and regularization strength
# [中文] 使用验证集来调整学习率和正则化强度

from cs231n.classifiers.linear_classifier import Softmax

learning_rates = [1e-7, 1e-6]
regularization_strengths = [5e5, 5e6]

results = {}
best_val = -1
best_softmax = None

################################################################################
# TODO:                                                                        #
# Use the validation set to set the learning rate and regularization strength. #
# This should be identical to the validation that you did for the Softmax; save#
# the best trained classifer in best_softmax. If you carefully tune the model, #
# you should be able to get accuracy of above 0.42 on the validation set.      #
################################################################################
# [中文] TODO: 使用校验集来设定学习率和校正强度。 这应该与您为 Softmax 所做的校验相同; 保存 best_softmax 中受过最佳训练的类比器。
# 如果您仔细调整模型, 您应该能够在校验集上取得 0.42 以上准确度。


# Print out results.
# [中文] 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved: %f' % best_val)

In [ ]:
# Evaluate your trained Softmax on the test set: you should be able to get at least 0.42
# [中文] 在测试集上评价您受过训练的软体轴: 您应该能够至少获得 0.42
y_test_pred = best_softmax.predict(X_test_feats)
test_accuracy = np.mean(y_test == y_test_pred)
print(test_accuracy)

In [ ]:
# Save best softmax model
# [中文] 保存最好的软体模型
best_softmax.save("best_softmax_features.npy")

In [ ]:
# An important way to gain intuition about how an algorithm works is to
# visualize the mistakes that it makes. In this visualization, we show examples
# of images that are misclassified by our current system. The first column
# shows images that our system labeled as "plane" but whose true label is
# something other than "plane".
# [中文] 获得关于算法如何运作的直觉的一个重要方法就是直观地描述它造成的错误。 在这种可视化中,我们展示了被我们当前系统错误分类的图像实例。
# 第一栏显示的是被我们系统标为“飞机”但真正标签不是“飞机”的图像。

examples_per_class = 8
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for cls, cls_name in enumerate(classes):
    idxs = np.where((y_test != cls) & (y_test_pred == cls))[0]
    idxs = np.random.choice(idxs, examples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt.subplot(examples_per_class, len(classes), i * len(classes) + cls + 1)
        plt.imshow(X_test[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls_name)
plt.show()

### Inline question 1:
Describe the misclassification results that you see. Do they make sense?


$\color{blue}{\textit Your Answer:}$





---

**中文对照（参考译文，不含解答）**

### 随堂问题 1

描述你观察到的错误分类结果。这些结果是否合理？

**你的回答：** [请自行填写]

## Neural Network on image features
Earlier in this assigment we saw that training a two-layer neural network on raw pixels achieved better classification performance than linear classifiers on raw pixels. In this notebook we have seen that linear classifiers on image features outperform linear classifiers on raw pixels.

For completeness, we should also try training a neural network on image features. This approach should outperform all previous approaches: you should easily be able to achieve over 55% classification accuracy on the test set; our best model achieves about 60% classification accuracy.

---

**中文对照（参考译文，不含解答）**

## 图像特征神经网络
稍早一点,我们发现对两层神经网络进行生像素训练比对生像素进行线性分类的结果要好。 在这本笔记本中,我们看到了图像线性分类的特征优于生像素的线性分类。

为了完整起见, 我们还应该尝试对一个神经网络进行图像特征训练。 这个方法应该比以往所有方法都好: 您应该很容易在测试集上达到 55% 分类准确率; 我们的最佳模型可以达到 60% 分类精确度。

In [ ]:
# Preprocessing: Remove the bias dimension
# Make sure to run this cell only ONCE
# [中文] 预处理: 删除偏差维度以确保只运行此单元格 ONCE
print(X_train_feats.shape)
X_train_feats = X_train_feats[:, :-1]
X_val_feats = X_val_feats[:, :-1]
X_test_feats = X_test_feats[:, :-1]

print(X_train_feats.shape)

In [ ]:
from cs231n.classifiers.fc_net import TwoLayerNet
from cs231n.solver import Solver

input_dim = X_train_feats.shape[1]
hidden_dim = 500
num_classes = 10

data = {
    'X_train': X_train_feats,
    'y_train': y_train,
    'X_val': X_val_feats,
    'y_val': y_val,
    'X_test': X_test_feats,
    'y_test': y_test,
}

net = TwoLayerNet(input_dim, hidden_dim, num_classes)
best_net = None

################################################################################
# TODO: Train a two-layer neural network on image features. You may want to    #
# cross-validate various parameters as in previous sections. Store your best   #
# model in the best_net variable.                                              #
################################################################################
# [中文] TODO: 对图像特性进行双层神经网络训练。 您可能想要像前几节一样交叉校验各种参数。 将您的最佳模型保存在 best_net 变量中。


In [ ]:
# Run your best neural net classifier on the test set. You should be able
# to get more than 58% accuracy. It is also possible to get >60% accuracy
# with careful tuning.
# [中文] 在测试集上运行您最好的神经网分类器。 您应该能够获得大于 58% 的准确率。 也可以在仔细调试的情况下获取 > 60% 的精确度。

y_test_pred = np.argmax(best_net.loss(data['X_test']), axis=1)
test_acc = (y_test_pred == data['y_test']).mean()
print(test_acc)

In [ ]:
# Save best model
# [中文] 保存最佳模型
best_net.save("best_two_layer_net_features.npy")